# Build LIMUC metadata + image manifest
Preparation pipeline for LIMUC UC severity benchmarks (Mayo Endoscopic Score). This notebook builds:
- `metadata_raw.csv`, `metadata_enriched.csv`
- `label_map.csv`
- image `manifest.csv`
- deterministic train/val/test splits (patient-level if available)


In [1]:
# Performance config
import os
CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)


In [2]:
import os
import re
import json
import hashlib
import random
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split, GroupShuffleSplit


In [3]:
# =====================
# Config
# =====================

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "Datasets" / "LIMUC").exists() or (p / "Prototyping_reformat").exists():
            return p
    return start

# Default extracted dataset location (relative to repo root)
REPO_ROOT = find_repo_root()
DEFAULT_DATASET_ROOT = REPO_ROOT / "Datasets" / "LIMUC"
DATASET_ROOT = Path(os.getenv("LIMUC_DATASET_ROOT", str(DEFAULT_DATASET_ROOT))).expanduser()

# Choose data variant:
# - "trainval_test": use train_and_validation_sets + test_set directories
# - "patient_based": use patient_based_classified_images (patient_id folders)
# - "folder_labels": generic folder-by-label scan
DATA_VARIANT = os.getenv("LIMUC_DATA_VARIANT", "trainval_test")

TRAINVAL_DIR = Path(os.getenv("LIMUC_TRAINVAL_DIR", str(DATASET_ROOT / "train_and_validation_sets"))).expanduser()
TEST_DIR = Path(os.getenv("LIMUC_TEST_DIR", str(DATASET_ROOT / "test_set"))).expanduser()
PATIENT_DIR = Path(os.getenv("LIMUC_PATIENT_DIR", str(DATASET_ROOT / "patient_based_classified_images"))).expanduser()

# If you already have a metadata file, set it here.
# Supported: .csv, .tsv, .json, .jsonl, .xlsx
RAW_METADATA_PATH = Path(os.getenv("LIMUC_METADATA", "")) if os.getenv("LIMUC_METADATA", "") else None

# Column names in metadata (edit to match your file)
IMAGE_PATH_COL = os.getenv("LIMUC_IMAGE_PATH_COL", "image_path")
LABEL_COL = os.getenv("LIMUC_LABEL_COL", "label")  # e.g., "mes", "score"
PATIENT_COL = os.getenv("LIMUC_PATIENT_COL", "patient_id")
SPLIT_COL = os.getenv("LIMUC_SPLIT_COL", "split")

# Generic folder scan fallback
USE_FOLDER_LABELS = os.getenv("LIMUC_USE_FOLDER_LABELS", "0") == "1"
IMAGE_ROOT = Path(os.getenv("LIMUC_IMAGE_ROOT", str(DATASET_ROOT))).expanduser()
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# Splitting behavior
USE_PREDEFINED_SPLIT = os.getenv("LIMUC_USE_PREDEFINED_SPLIT", "1") == "1"
SPLIT_BY_PATIENT = os.getenv("LIMUC_SPLIT_BY_PATIENT", "1") == "1"
SPLIT_SEED = int(os.getenv("LIMUC_SPLIT_SEED", "42"))
TRAIN_FRAC = float(os.getenv("LIMUC_TRAIN_FRAC", "0.8"))
VAL_FRAC = float(os.getenv("LIMUC_VAL_FRAC", "0.1"))
TEST_FRAC = float(os.getenv("LIMUC_TEST_FRAC", "0.1"))

# Label ordering (important for ordinal metrics). If empty, will infer.
# Example: ["0", "1", "2", "3"] or [0,1,2,3]
LABEL_ORDER = json.loads(os.getenv("LIMUC_LABEL_ORDER", "[0, 1, 2, 3]"))

# Path normalization / copying
COPY_IMAGES = os.getenv("LIMUC_COPY_IMAGES", "0") == "1"  # copy into out/images
READ_IMAGE_SIZES = os.getenv("LIMUC_READ_IMAGE_SIZES", "1") == "1"

OUT_ROOT = Path("./out")
IMAGES_DIR = OUT_ROOT / "images"
META_DIR = OUT_ROOT / "metadata"
MANIFEST_DIR = OUT_ROOT / "manifests"
SPLITS_DIR = OUT_ROOT / "splits"
for d in [IMAGES_DIR, META_DIR, MANIFEST_DIR, SPLITS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RAW_META_CSV = META_DIR / "metadata_raw.csv"
ENRICHED_META_CSV = META_DIR / "metadata_enriched.csv"
LABEL_MAP_CSV = META_DIR / "label_map.csv"
IMAGE_MANIFEST_CSV = MANIFEST_DIR / "image_manifest.csv"
SPLIT_HASH_TXT = META_DIR / "split_hash.txt"

print("Repo root:", REPO_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Data variant:", DATA_VARIANT)
print("Train/val dir:", TRAINVAL_DIR)
print("Test dir:", TEST_DIR)
print("Patient dir:", PATIENT_DIR)
print("Metadata path:", RAW_METADATA_PATH)
print("Image root:", IMAGE_ROOT)
print("Output root:", OUT_ROOT)


Dataset root: /home/arcturus/Desktop/thesis/rag-vqa-medical/Datasets/LIMUC
Data variant: trainval_test
Train/val dir: /home/arcturus/Desktop/thesis/rag-vqa-medical/Datasets/LIMUC/train_and_validation_sets
Test dir: /home/arcturus/Desktop/thesis/rag-vqa-medical/Datasets/LIMUC/test_set
Patient dir: /home/arcturus/Desktop/thesis/rag-vqa-medical/Datasets/LIMUC/patient_based_classified_images
Metadata path: None
Image root: /home/arcturus/Desktop/thesis/rag-vqa-medical/Datasets/LIMUC
Output root: out


In [4]:
# =====================
# Helpers
# =====================
def _read_metadata(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Metadata file not found: {path}")
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".tsv":
        return pd.read_csv(path, sep="	")
    if suffix in {".json", ".jsonl"}:
        return pd.read_json(path, lines=suffix == ".jsonl")
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported metadata extension: {suffix}")


def _infer_patient_id(path_str: str) -> str | None:
    if not path_str:
        return None
    m = re.search(r"patient[_-]?(\d+)", str(path_str), flags=re.IGNORECASE)
    if m:
        return m.group(1)
    return None


def _build_from_folders(image_root: Path) -> pd.DataFrame:
    rows = []
    if not image_root.exists():
        raise FileNotFoundError(f"Image root does not exist: {image_root}")
    for label_dir in sorted([p for p in image_root.iterdir() if p.is_dir()]):
        label_name = label_dir.name
        for img_path in label_dir.rglob("*"):
            if img_path.suffix.lower() in IMAGE_EXTS:
                rows.append({
                    "image_path": str(img_path),
                    "label_name": label_name,
                    "patient_id": _infer_patient_id(str(img_path)),
                })
    if not rows:
        raise ValueError("No images found when scanning folders. Check IMAGE_ROOT and extensions.")
    return pd.DataFrame(rows)


def _walk_label_dirs(root: Path, split_name: str | None = None, patient_id: str | None = None):
    rows = []
    if not root.exists():
        return rows
    for label_dir in sorted([p for p in root.iterdir() if p.is_dir()]):
        label_name = label_dir.name
        for img_path in label_dir.rglob("*"):
            if img_path.suffix.lower() in IMAGE_EXTS:
                rows.append({
                    "image_path": str(img_path),
                    "label_name": label_name,
                    "split": split_name,
                    "patient_id": patient_id or _infer_patient_id(str(img_path)),
                })
    return rows


def _build_from_trainval_test(trainval_dir: Path, test_dir: Path) -> pd.DataFrame:
    rows = []
    rows.extend(_walk_label_dirs(trainval_dir, split_name=None))
    rows.extend(_walk_label_dirs(test_dir, split_name="test"))
    if not rows:
        raise ValueError("No images found in train/val/test directories. Check paths.")
    return pd.DataFrame(rows)


def _build_from_patient_folders(patient_root: Path) -> pd.DataFrame:
    rows = []
    if not patient_root.exists():
        raise FileNotFoundError(f"Patient root does not exist: {patient_root}")
    for patient_dir in sorted([p for p in patient_root.iterdir() if p.is_dir()]):
        patient_id = patient_dir.name
        rows.extend(_walk_label_dirs(patient_dir, split_name=None, patient_id=patient_id))
    if not rows:
        raise ValueError("No images found in patient_based_classified_images.")
    return pd.DataFrame(rows)


def _normalize_image_path(p: str) -> str:
    if p is None or str(p).strip() == "":
        return ""
    p = Path(str(p))
    if p.is_absolute():
        return str(p)
    # Try IMAGE_ROOT first, then DATASET_ROOT
    cand = (IMAGE_ROOT / p).resolve()
    if cand.exists():
        return str(cand)
    cand = (DATASET_ROOT / p).resolve()
    return str(cand)


def _safe_img_id(path: str, idx: int) -> str:
    p = Path(path)
    stem = p.stem
    if stem:
        return stem
    return f"img_{idx:06d}"


def _normalize_label(val) -> str:
    if pd.isna(val):
        return ""
    s = str(val).strip()
    if s == "":
        return ""
    # Extract a Mayo score digit if present (0-3)
    m = re.search(r"(?<!\d)([0-3])(?!\d)", s)
    if m:
        return m.group(1)
    return s


def _infer_label_map(labels: pd.Series) -> Tuple[dict, dict]:
    # Use provided label order if possible
    if LABEL_ORDER:
        ordered = [str(x) for x in LABEL_ORDER]
        present = [x for x in ordered if x in set(labels)]
        missing = [x for x in set(labels) if x not in set(present)]
        try:
            missing = sorted(missing, key=lambda x: int(x))
        except Exception:
            missing = sorted(missing)
        merged = present + missing
        if merged:
            id_to_name = {i: name for i, name in enumerate(merged)}
            return id_to_name, {v: k for k, v in id_to_name.items()}
    # Fallback: numeric sort if possible
    unique = sorted(set(labels))
    try:
        unique_sorted = sorted(unique, key=lambda x: int(x))
    except Exception:
        unique_sorted = sorted(unique)
    id_to_name = {i: name for i, name in enumerate(unique_sorted)}
    return id_to_name, {v: k for k, v in id_to_name.items()}


def _compute_split_hash(df: pd.DataFrame) -> str:
    key = df[["img_id", "split"]].astype(str).sort_values(["img_id", "split"])
    h = hashlib.sha256()
    for row in key.itertuples(index=False):
        h.update(f"{row.img_id}|{row.split}".encode())
    return h.hexdigest()


def _split_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().reset_index(drop=True)

    # Normalize any existing split labels
    if "split" in df.columns and df["split"].notna().any() and USE_PREDEFINED_SPLIT:
        def _norm_split(s):
            if s is None or (isinstance(s, float) and pd.isna(s)):
                return None
            s = str(s).strip().lower()
            s = {"validation": "val", "valid": "val", "dev": "val"}.get(s, s)
            if s in {"train", "val", "test"}:
                return s
            return None

        df["split"] = df["split"].apply(_norm_split)

        assigned = df[df["split"].notna()].copy()
        unassigned = df[df["split"].isna()].copy()

        if len(unassigned) == 0:
            return df

        has_test_assigned = (assigned["split"] == "test").any()
        df_to_split = unassigned
    else:
        assigned = pd.DataFrame(columns=df.columns)
        df_to_split = df
        has_test_assigned = False

    # Otherwise create splits for df_to_split
    if "patient_id" in df_to_split.columns and df_to_split["patient_id"].notna().any() and SPLIT_BY_PATIENT:
        groups = df_to_split["patient_id"].astype(str)
        if has_test_assigned:
            # Only create train/val when an official test set exists
            gss = GroupShuffleSplit(n_splits=1, test_size=VAL_FRAC / max(TRAIN_FRAC + VAL_FRAC, 1e-8), random_state=SPLIT_SEED)
            train_idx, val_idx = next(gss.split(df_to_split, groups=groups))
            train_df = df_to_split.iloc[train_idx].copy()
            val_df = df_to_split.iloc[val_idx].copy()
            test_df = df_to_split.iloc[0:0].copy()
        else:
            gss = GroupShuffleSplit(n_splits=1, test_size=VAL_FRAC + TEST_FRAC, random_state=SPLIT_SEED)
            train_idx, temp_idx = next(gss.split(df_to_split, groups=groups))
            train_df = df_to_split.iloc[train_idx].copy()
            temp_df = df_to_split.iloc[temp_idx].copy()

            # Split temp into val/test
            temp_groups = temp_df["patient_id"].astype(str)
            test_ratio = TEST_FRAC / max(VAL_FRAC + TEST_FRAC, 1e-8)
            gss2 = GroupShuffleSplit(n_splits=1, test_size=test_ratio, random_state=SPLIT_SEED)
            val_idx, test_idx = next(gss2.split(temp_df, groups=temp_groups))
            val_df = temp_df.iloc[val_idx].copy()
            test_df = temp_df.iloc[test_idx].copy()
    else:
        y = df_to_split["label_name"].astype(str)
        if has_test_assigned:
            try:
                train_df, val_df = train_test_split(
                    df_to_split, test_size=VAL_FRAC / max(TRAIN_FRAC + VAL_FRAC, 1e-8), stratify=y, random_state=SPLIT_SEED
                )
            except Exception:
                train_df, val_df = train_test_split(
                    df_to_split, test_size=VAL_FRAC / max(TRAIN_FRAC + VAL_FRAC, 1e-8), random_state=SPLIT_SEED
                )
            test_df = df_to_split.iloc[0:0].copy()
        else:
            try:
                train_df, temp_df = train_test_split(
                    df_to_split, test_size=VAL_FRAC + TEST_FRAC, stratify=y, random_state=SPLIT_SEED
                )
            except Exception:
                train_df, temp_df = train_test_split(
                    df_to_split, test_size=VAL_FRAC + TEST_FRAC, random_state=SPLIT_SEED
                )
            y_temp = temp_df["label_name"].astype(str)
            test_ratio = TEST_FRAC / max(VAL_FRAC + TEST_FRAC, 1e-8)
            try:
                val_df, test_df = train_test_split(
                    temp_df, test_size=test_ratio, stratify=y_temp, random_state=SPLIT_SEED
                )
            except Exception:
                val_df, test_df = train_test_split(
                    temp_df, test_size=test_ratio, random_state=SPLIT_SEED
                )

    train_df["split"] = "train"
    val_df["split"] = "val"
    if len(test_df) > 0:
        test_df["split"] = "test"

    combined = pd.concat([assigned, train_df, val_df, test_df], ignore_index=True)
    return combined


In [5]:
# =====================
# Load / build raw metadata
# =====================
if RAW_METADATA_PATH is not None and RAW_METADATA_PATH.exists():
    df_raw = _read_metadata(RAW_METADATA_PATH)
elif DATA_VARIANT == "patient_based":
    df_raw = _build_from_patient_folders(PATIENT_DIR)
elif DATA_VARIANT == "trainval_test":
    df_raw = _build_from_trainval_test(TRAINVAL_DIR, TEST_DIR)
elif USE_FOLDER_LABELS or DATA_VARIANT == "folder_labels":
    df_raw = _build_from_folders(IMAGE_ROOT)
else:
    raise ValueError(
        "No metadata provided. Set LIMUC_METADATA or choose DATA_VARIANT."
    )

# Normalize columns
if IMAGE_PATH_COL in df_raw.columns:
    image_col = IMAGE_PATH_COL
elif "image_path" in df_raw.columns:
    image_col = "image_path"
else:
    raise KeyError(f"Missing image path column: {IMAGE_PATH_COL}")

if LABEL_COL in df_raw.columns:
    label_col = LABEL_COL
elif "label_name" in df_raw.columns:
    label_col = "label_name"
else:
    raise KeyError(f"Missing label column: {LABEL_COL}")

# Build standard frame
df = pd.DataFrame({
    "image_path": df_raw[image_col].astype(str),
    "label_name": df_raw[label_col],
})

# Patient ID (if present)
if PATIENT_COL in df_raw.columns:
    df["patient_id"] = df_raw[PATIENT_COL]
elif "patient_id" in df_raw.columns:
    df["patient_id"] = df_raw["patient_id"]

# Split (if present)
if SPLIT_COL in df_raw.columns:
    df["split"] = df_raw[SPLIT_COL]
elif "split" in df_raw.columns:
    df["split"] = df_raw["split"]

# Normalize paths and labels
df["image_path"] = df["image_path"].apply(_normalize_image_path)
df["label_name"] = df["label_name"].apply(_normalize_label)

# Infer patient_id from filename if missing
if "patient_id" not in df.columns or df["patient_id"].isna().all():
    df["patient_id"] = df["image_path"].apply(_infer_patient_id)

# Drop missing
df = df[df["image_path"].astype(str).str.len() > 0]
df = df[df["label_name"].astype(str).str.len() > 0]
df = df.reset_index(drop=True)

# Build img_id
df["img_id"] = [
    _safe_img_id(p, i) for i, p in enumerate(df["image_path"].tolist())
]

# Ensure unique img_id
if df["img_id"].duplicated().any():
    df["img_id"] = [f"{iid}_{i:06d}" for i, iid in enumerate(df["img_id"].tolist())]

# Optionally read image sizes
if READ_IMAGE_SIZES:
    heights, widths = [], []
    for p in df["image_path"].tolist():
        try:
            with Image.open(p) as im:
                w, h = im.size
        except Exception:
            w, h = None, None
        widths.append(w)
        heights.append(h)
    df["orig_height"] = heights
    df["orig_width"] = widths

print("Raw rows:", len(df))
print(df.head())


Raw rows: 11276
                                          image_path label_name patient_id  \
0  /home/arcturus/Desktop/thesis/rag-vqa-medical/...          0        185   
1  /home/arcturus/Desktop/thesis/rag-vqa-medical/...          0        307   
2  /home/arcturus/Desktop/thesis/rag-vqa-medical/...          0        337   
3  /home/arcturus/Desktop/thesis/rag-vqa-medical/...          0        489   
4  /home/arcturus/Desktop/thesis/rag-vqa-medical/...          0        525   

  split                    img_id  orig_height  orig_width  
0  None  UC_patient_185_10_000000          288         352  
1  None  UC_patient_307_34_000001          288         352  
2  None  UC_patient_337_13_000002          288         352  
3  None  UC_patient_489_27_000003          288         352  
4  None  UC_patient_525_53_000004          288         352  


In [6]:
# =====================
# Split + label mapping
# =====================
df = _split_df(df)

id_to_name, name_to_id = _infer_label_map(df["label_name"].astype(str))
df["label_id"] = df["label_name"].astype(str).map(name_to_id)

# Save label map
label_map = pd.DataFrame({
    "label_id": list(id_to_name.keys()),
    "label_name": list(id_to_name.values()),
})

# Save raw/enriched metadata
df_raw_out = df[[
    "split",
    "img_id",
    "image_path",
    "label_id",
    "label_name",
]].copy()
if "patient_id" in df.columns:
    df_raw_out["patient_id"] = df["patient_id"]
if "orig_height" in df.columns:
    df_raw_out["orig_height"] = df["orig_height"]
if "orig_width" in df.columns:
    df_raw_out["orig_width"] = df["orig_width"]

df_enriched = df_raw_out.copy()

# Compute split hash
split_hash = _compute_split_hash(df_enriched)


In [7]:
# =====================
# Optional: copy images into out/images/<split>/
# =====================
if COPY_IMAGES:
    from shutil import copy2

    new_paths = []
    for row in df_enriched.itertuples(index=False):
        src = Path(row.image_path)
        if not src.exists():
            new_paths.append(str(src))
            continue
        dst = IMAGES_DIR / row.split
        dst.mkdir(parents=True, exist_ok=True)
        out_path = dst / f"{row.img_id}{src.suffix.lower()}"
        if not out_path.exists():
            copy2(src, out_path)
        # Store relative path so downstream can resolve from 0_dataset_prep
        rel = out_path.relative_to(Path("."))
        new_paths.append(str(rel))
    df_enriched["image_path"] = new_paths

# Save to disk
df_raw_out.to_csv(RAW_META_CSV, index=False)
df_enriched.to_csv(ENRICHED_META_CSV, index=False)
label_map.to_csv(LABEL_MAP_CSV, index=False)

# Manifest
manifest = df_enriched[["split", "img_id", "image_path"]].copy()
manifest["exists"] = manifest["image_path"].apply(lambda p: Path(p).exists())
manifest.to_csv(IMAGE_MANIFEST_CSV, index=False)

# Split lists
for split_name in ["train", "val", "test"]:
    split_ids = df_enriched.loc[df_enriched["split"] == split_name, "img_id"].tolist()
    (SPLITS_DIR / f"{split_name}.txt").write_text("\n".join(split_ids))

# Split hash
SPLIT_HASH_TXT.write_text(split_hash)

print("Saved:")
print("-", RAW_META_CSV)
print("-", ENRICHED_META_CSV)
print("-", LABEL_MAP_CSV)
print("-", IMAGE_MANIFEST_CSV)
print("-", SPLIT_HASH_TXT)


Saved:
- out/metadata/metadata_raw.csv
- out/metadata/metadata_enriched.csv
- out/metadata/label_map.csv
- out/manifests/image_manifest.csv
- out/metadata/split_hash.txt


In [8]:
# =====================
# Quick checks
# =====================
print("Rows by split:")
print(df_enriched["split"].value_counts())

print("Label distribution:")
print(df_enriched["label_name"].value_counts())

print("Split hash:", split_hash)


Rows by split:
split
train    8669
test     1686
val       921
Name: count, dtype: int64
Label distribution:
label_name
0    6105
1    3052
2    1254
3     865
Name: count, dtype: int64
Split hash: d71d3864f86c77641c029b050ab26b74e62f8425940c4131fd708a964b78008b
